# LangGraph Level 2B — HITL · Streaming · Persistent Memory

Patterns covered:
1. **`interrupt_before`** — pause the graph before executing an action, ask for human confirmation
2. **Streaming** — receive outputs per node in real time (don't wait for everything to finish)
3. **SQLite Checkpointer** — persistent memory across different sessions (not just in RAM)

In [16]:
from dotenv import load_dotenv
load_dotenv()

True

In [17]:
from typing import Literal, Annotated
from IPython.display import Image, display

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

---
## 1. `interrupt_before` — Human-in-the-Loop

**What is it?** The graph pauses right **before** executing a specific node.
The human can review the state and decide whether to continue, modify, or cancel.

**When to use it:** Before irreversible actions (sending email, executing code, calling external API).

```
[plan_action] → PAUSE here ← human reviews
                    ↓ confirms
             [execute_action] → END
```

**Requires a checkpointer** — the state is persisted while waiting for confirmation.

In [ ]:
class ActionState(MessagesState):
    proposed_action: str
    action_result: str
    approved: bool

def plan_action(state: ActionState) -> ActionState:
    """Generates a proposed action based on user input."""
    user_request = state["messages"][-1].content
    response = llm.invoke([
        SystemMessage(content="You are an AI assistant. The user wants to execute something. Describe exactly what you would do in a short sentence."),
        HumanMessage(content=user_request)
    ])
    return {
        "proposed_action": response.content,
        "messages": [AIMessage(content=f"Proposed action: {response.content}")]
    }

def execute_action(state: ActionState) -> ActionState:
    """Executes the action. Only reaches here if the human approved."""
    result = f"✅ Executed: {state['proposed_action']}"
    return {
        "action_result": result,
        "messages": [AIMessage(content=result)]
    }

memory = InMemorySaver()

builder = StateGraph(ActionState)
builder.add_node("plan_action", plan_action)
builder.add_node("execute_action", execute_action)
builder.add_edge(START, "plan_action")
builder.add_edge("plan_action", "execute_action")
builder.add_edge("execute_action", END)

# interrupt_before pauses the graph BEFORE execute_action
hitl_graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["execute_action"]  # <-- HITL here
)

display(Image(hitl_graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "hitl-demo-1"}}

# --- PHASE 1: The graph plans and pauses ---
print("=== PHASE 1: Run until interrupt ===")
result = hitl_graph.invoke(
    {"messages": [HumanMessage(content="Send an email to the whole team notifying them about today's deploy")]},
    config
)

print("State paused at:", hitl_graph.get_state(config).next)
print("Proposed action:", result.get("proposed_action"))
print()

# --- PHASE 2: The human reviews and decides to continue ---
print("=== PHASE 2: Human approves → resume ===")
# To resume, invoke with None as input (uses the saved state)
final_result = hitl_graph.invoke(None, config)
print("Result:", final_result.get("action_result"))

In [ ]:
# --- Variant: modify the state before resuming ---
config2 = {"configurable": {"thread_id": "hitl-demo-2"}}

result2 = hitl_graph.invoke(
    {"messages": [HumanMessage(content="Delete all temporary files from the server")]},
    config2
)

print("Paused at:", hitl_graph.get_state(config2).next)
print("Proposed action:", result2.get("proposed_action"))

print("The human decides to MODIFY the action before approving:")

# Manually update the state before resuming
hitl_graph.update_state(
    config2,
    {"proposed_action": "List temporary files (read-only, do not delete)"}
)

final = hitl_graph.invoke(None, config2)
print("Executed (modified version):", final.get("action_result"))

---
## 2. Streaming — outputs per node in real time

**What is it?** Instead of waiting for the graph to finish (`invoke`), `stream` emits chunks
as each node generates output. Essential for real-time UX.

**Streaming modes:**
- `stream_mode="messages"` → text chunks from the LLM (token by token)
- `stream_mode="values"` → complete state after each node
- `stream_mode="updates"` → only the delta (what changed) at each node

```python
# invoke  → waits for everything → returns final result
# stream  → emits as it progresses → better UX
```

In [ ]:
class WritingState(MessagesState):
    topic: str
    outline: str
    content: str

def create_outline(state: WritingState) -> WritingState:
    """Node 1: generates the outline."""
    response = llm.invoke([
        SystemMessage(content="Create a brief 3-point outline for a technical article."),
        HumanMessage(content=f"Topic: {state['topic']}")
    ])
    return {"outline": response.content}

def write_content(state: WritingState) -> WritingState:
    """Node 2: develops the content."""
    response = llm.invoke([
        SystemMessage(content="Develop the article based on the given outline. Max 150 words."),
        HumanMessage(content=f"Outline:\n{state['outline']}")
    ])
    return {"content": response.content}

builder = StateGraph(WritingState)
builder.add_node("create_outline", create_outline)
builder.add_node("write_content", write_content)
builder.add_edge(START, "create_outline")
builder.add_edge("create_outline", "write_content")
builder.add_edge("write_content", END)
writing_graph = builder.compile()

In [ ]:
print("=== stream_mode='updates' — delta per node ===")
print()

for chunk in writing_graph.stream(
    {"topic": "Why use graphs to orchestrate AI agents"},
    stream_mode="updates"
):
    node_name = list(chunk.keys())[0]
    node_output = chunk[node_name]
    print(f"[{node_name}] generated:")
    for key, value in node_output.items():
        if isinstance(value, str):
            print(f"  {key}: {value[:120]}...")
    print()

In [ ]:
print("=== stream_mode='messages' — token by token ===")
print("(this is how you create the typewriter effect in chat UIs)")
print()

for chunk, metadata in writing_graph.stream(
    {"topic": "Persistent memory in multi-agent systems"},
    stream_mode="messages"
):
    # chunk is an AIMessageChunk when the LLM is generating
    if hasattr(chunk, 'content') and chunk.content:
        print(chunk.content, end="", flush=True)

print("\n\n[stream completed]")

---
## 3. SQLite Checkpointer — persistent memory across sessions

**What is it?** `InMemorySaver` stores state in RAM (lost when the process closes).
`SqliteSaver` stores on disk → the state survives across sessions.

**Checkpointer table:**

| Checkpointer | Storage | Persists | Use case |
|---|---|---|---|
| `InMemorySaver` | RAM | No | Dev / testing |
| `SqliteSaver` | `.db` file | Yes | Local / prototype |
| `PostgresSaver` | PostgreSQL | Yes | Production |

**Why does it matter?** With SQLite, if you restart the notebook the conversation history is still there.

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

class ChatState(MessagesState):
    session_name: str

def chat_with_memory(state: ChatState) -> ChatState:
    """Conversational agent — remembers the history across sessions."""
    response = llm.invoke([
        SystemMessage(content="You are a technical assistant. You remember everything discussed in this session."),
        *state["messages"]
    ])
    return {"messages": [response]}

builder = StateGraph(ChatState)
builder.add_node("chat", chat_with_memory)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

In [ ]:
# SESSION 1 — create SQLite connection and save history
DB_PATH = "./chat_memory.db"
config = {"configurable": {"thread_id": "user-pedro-001"}}

# from_conn_string receives the file path — handles the connection internally
with SqliteSaver.from_conn_string(DB_PATH) as sqlite_memory:
    graph_sqlite = builder.compile(checkpointer=sqlite_memory)
    
    print("=== SESSION 1 — First messages ===")
    
    # Turn 1
    r1 = graph_sqlite.invoke(
        {"messages": [HumanMessage(content="Hi, I'm learning LangGraph")]},
        config
    )
    print(f"Human: Hi, I'm learning LangGraph")
    print(f"AI: {r1['messages'][-1].content[:200]}")
    print()
    
    # Turn 2
    r2 = graph_sqlite.invoke(
        {"messages": [HumanMessage(content="What is the most important concept I should understand first?")]},
        config
    )
    print(f"Human: What is the most important concept...?")
    print(f"AI: {r2['messages'][-1].content[:200]}")
    print()
    print(f"Messages in history: {len(r2['messages'])}")
    print("[SQLite saved to disk — close the kernel and re-run the next block]")

In [ ]:
# SESSION 2 — new connection, same data (simulating a process restart)
config = {"configurable": {"thread_id": "user-pedro-001"}}  # same thread_id

with SqliteSaver.from_conn_string(DB_PATH) as sqlite_memory2:
    graph_sqlite2 = builder.compile(checkpointer=sqlite_memory2)
    
    print("=== SESSION 2 — New connection, history recovered ===")
    
    r3 = graph_sqlite2.invoke(
        {"messages": [HumanMessage(content="What were we talking about before?")]},
        config
    )
    print(f"Human: What were we talking about before?")
    print(f"AI: {r3['messages'][-1].content[:300]}")
    print()
    print(f"Total messages in history (includes previous session): {len(r3['messages'])}")

In [ ]:
# Comparison: same graph, different thread_id = isolated histories
config_user_a = {"configurable": {"thread_id": "user-alice"}}
config_user_b = {"configurable": {"thread_id": "user-bob"}}

with SqliteSaver.from_conn_string(DB_PATH) as mem:
    g = builder.compile(checkpointer=mem)
    
    g.invoke({"messages": [HumanMessage(content="My name is Alice")]}, config_user_a)
    g.invoke({"messages": [HumanMessage(content="My name is Bob")]}, config_user_b)
    
    ra = g.invoke({"messages": [HumanMessage(content="What is my name?")]}, config_user_a)
    rb = g.invoke({"messages": [HumanMessage(content="What is my name?")]}, config_user_b)
    
    print("Alice asks 'What is my name?':")
    print(" ", ra['messages'][-1].content[:150])
    print()
    print("Bob asks 'What is my name?':")
    print(" ", rb['messages'][-1].content[:150])

---
## Takeaways — N2B

| Pattern | Mechanism | When to use it |
|---------|-----------|----------------|
| `interrupt_before` | Pauses the graph before a specific node | Before irreversible actions |
| `update_state` | Modifies the state while the graph is paused | For the human to correct the proposed action |
| `stream(mode='updates')` | Delta per node | Debug / real-time logs |
| `stream(mode='messages')` | Token by token | Chat UI with typewriter effect |
| `SqliteSaver` | Saves to disk | Memory across sessions (local / prototype) |
| `thread_id` | Isolates histories | One thread per user/conversation |

**Next:** `N2C_patterns_advanced.ipynb` — Orchestrator-Worker · Evaluator-Optimizer